In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os
import numpy as np
import scipy as sc
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.model_selection import LeaveOneOut, StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

### TRAINING / EVALUATION

In [ ]:
def train(X, y):
    models = {
        "Logistic Regression (L1)": Pipeline([
            ("scaler", StandardScaler()),
            ("select", SelectKBest(f_classif, k=min(50, X.shape[1]))),
            ("clf", LogisticRegression(
                penalty="l1", solver="saga", C=0.1, max_iter=1000, random_state=42
            ))
        ]),
        "Logistic Regression (L2)": Pipeline([
            ("scaler", StandardScaler()),
            ("select", SelectKBest(f_classif, k=min(50, X.shape[1]))),
            ("clf", LogisticRegression(
                penalty="l2", C=0.1, max_iter=1000, random_state=42
            ))
        ]),
        "Linear SVM": Pipeline([
            ("scaler", StandardScaler()),
            ("select", SelectKBest(f_classif, k=min(50, X.shape[1]))),
            ("clf", SVC(
                kernel="linear", C=0.1, probability=True, random_state=42
            ))
        ]),
        "RBF SVM": Pipeline([
            ("scaler", StandardScaler()),
            ("select", SelectKBest(f_classif, k=min(50, X.shape[1]))),
            ("clf", SVC(
                kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42
            ))
        ]),
        "Random Forest": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", RandomForestClassifier(
                n_estimators=20, max_depth=3, min_samples_split=3, random_state=42
            ))
        ]),
        "Gradient Boosting": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", GradientBoostingClassifier(
                n_estimators=20, max_depth=2, learning_rate=0.1, random_state=42
            ))
        ])
    }
    loo, res = LeaveOneOut(), {}
    for name, model in models.items():
        cv_scores = cross_val_score(model, X, y, cv=loo, scoring="accuracy")
        y_pred = cross_val_predict(model, X, y, cv=loo)
        res[name] = {
            "accuracy": np.mean(cv_scores),
            "std": np.std(cv_scores),
            "predictions": y_pred,
            "model": model
        }
    return res

In [ ]:
def printres(res):
    for name, model in res.items():
        print(name)
        print(classification_report(np.array(labels), model["predictions"]))
        print(confusion_matrix(np.array(labels), model["predictions"]), "\n")

### APPROACH 1 (FEATURE-BASED)

In [ ]:
def loaddir(dirname, only_fastballs=False):
    poses, labels = [], []
    classes = ["Curv", "Four", "Sink", "Slid"]
    for file in os.listdir(dirname):
        if os.path.isdir(file):
            continue
        if (arr := np.load(dirname + file)).ndim != 3:
            continue
        poses.append(arr)
        labels.append(
            file[:4] == "Four" if only_fastballs else classes.index(file[:4])
        )
    return poses, labels

In [ ]:
def normalize_sequence(seq):
    if seq.shape[1] > 5:
        shoulder_center = np.mean(seq[:, [5, 6], :], axis=1)
        hip_center = np.mean(seq[:, [11, 12], :], axis=1)
        torso_center = (shoulder_center + hip_center) / 2
        seq = seq - torso_center[:, np.newaxis, :]
    if seq.shape[1] > 12:
        shoulder_y = np.mean(seq[:, [5, 6], 1], axis=1)
        hip_y = np.mean(seq[:, [11, 12], 1], axis=1)
        torso_height = np.abs(shoulder_y - hip_y)
        scale = np.mean(torso_height)
        seq = seq / scale if scale > 0 else seq
    return seq

In [ ]:
def extract_features(seq):
    features = []
    n_frames, n_joints, _ = seq.shape
    
    ### (1) PER-JOINT STATISTICAL FEATURES
    for joint in range(n_joints):
        traj = seq[:, joint, :]
        features.extend(np.mean(traj, axis=0))
        features.extend(np.std(traj, axis=0))
        features.extend(np.ptp(traj, axis=0))
        if n_frames > 1:
            velocity = np.diff(traj, axis=0)
            features.extend(np.mean(velocity, axis=0))
            features.extend(np.std(velocity, axis=0))
            features.append(np.linalg.norm(traj[-1] - traj[0]))
        else:
            features.extend([0, 0, 0, 0, 0])
    
    ### (2) BODY-CENTER RELATIVE FEATURES
    if n_joints > 5:
        shoulder_center = np.mean(seq[:, [5, 6], :], axis=1)
        hip_center = np.mean(seq[:, [11, 12], :], axis=1)
        body_center = (shoulder_center + hip_center) / 2
        for joint in range(n_joints):
            rel_pos = seq[:, joint, :] - body_center
            features.extend(np.mean(rel_pos, axis=0))  # 2
            features.extend(np.std(rel_pos, axis=0))   # 2
    
    ### (3) JOINT-ANGLE FEATURES
    angle_definitions = [(5, 7, 9), (6, 8, 10), (5, 11, 13), (6, 12, 14), (5, 6, 8)]
    for a, b, c in angle_definitions:
        if max(a, b, c) < n_joints:
            v1 = seq[:, a, :] - seq[:, b, :]
            v2 = seq[:, c, :] - seq[:, b, :]
            dot = np.sum(v1 * v2, axis=1)
            norm = np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1)
            angle = np.arccos(np.clip(dot / (norm + 1e-6), -1, 1))
            features.extend([
                np.mean(angle),
                np.std(angle),
                np.ptp(angle),
                angle[0] if n_frames > 0 else 0,
                angle[-1] if n_frames > 0 else 0
            ])
    
    ### (4) TEMPORAL (FOURIER) FEATURES
    for joint in range(min(5, n_joints)):
        traj = seq[:, joint, 0]
        if n_frames > 3:
            fft_coef = np.abs(np.fft.fft(traj))[:3]
            features.extend(fft_coef)
        traj = seq[:, joint, 1]
        if n_frames > 3:
            fft_coef = np.abs(np.fft.fft(traj))[:3]
            features.extend(fft_coef)
    
    ### (5) OVERALL FEATURES)
    features.extend([
        np.mean(np.std(seq, axis=0)),
        np.max(np.linalg.norm(np.diff(seq, axis=0), axis=-1))
    ])
    return np.array(features)

In [ ]:
def create_feature_matrix(poses):
    featlist, featpad = [], []
    for seq in poses:
        feat = extract_features(seq)
        featlist.append(feat)
    maxlen = max(len(f) for f in featlist)
    for f in featlist:
        if len(f) < maxlen:
            f_pad = np.pad(f, (0, maxlen - len(f)), mode='constant')
        else:
            f_pad = f[:maxlen]
        featpad.append(f_pad)
    return np.array(featpad)

In [ ]:
poses, labels = loaddir("pitcher_clips/Tyler_Glasnow/keypoints/")
poses = [normalize_sequence(seq) for seq in poses]
X = create_feature_matrix(poses)
res = train(X, np.array(labels))

### APPROACH 2: ISHII'S IMPLEMENTATION

In [ ]:
def find_set_position_frame(pose_sequence):
    n_frames = len(pose_sequence)
    leg_angles, valid_frames = [], []
    hip, kicking_knee, kicking_ankle = 11, 12, 13
    
    for i, frame in enumerate(pose_sequence):
        if any(frame[n, 0] == 0 for n in (hip, kicking_knee, kicking_ankle)):
            leg_angles.append(180)
            continue
        
        hip_pt, knee, ankle = frame[hip], frame[kicking_knee], frame[kicking_ankle]
        thigh, calf = knee - hip_pt, ankle - knee
        
        if min(np.linalg.norm(thigh), np.linalg.norm(calf)) > 0:
            cos_angle = np.linalg.norm(thigh) * np.linalg.norm(calf)
            cos_angle = np.clip(np.dot(thigh, calf) / cos_angle, -1, 1)
            angle = np.arccos(cos_angle) * 180 / np.pi
        else:
            angle = 180
        leg_angles.append(angle)
        valid_frames.append(i)
    
    leg_angles = np.array(leg_angles)
    if len(valid_frames) == 0:
        return n_frames // 2
    
    min_angle_idx = valid_frames[np.argmin(leg_angles[valid_frames])]
    knee = pose_sequence[:, kicking_knee, 0]
    rolling_diff = np.zeros(n_frames)
    W = 5
    
    for i in range(W, n_frames - W):
        before = knee[i-W:i]
        after = knee[i:i+W]
        before_mean = np.mean(before[before > 0]) if np.any(before > 0) else 0
        after_mean = np.mean(after[after > 0]) if np.any(after > 0) else 0
        rolling_diff[i] = after_mean - before_mean
    
    set_frame_idx = None
    for i in range(min_angle_idx, W, -1):
        before_diff = np.mean(rolling_diff[i-W:i])
        after_diff = np.mean(rolling_diff[i:i+W])
        if before_diff < 0.001 and after_diff > 0.001:
            set_frame_idx = i
            break
    
    if set_frame_idx is None:
        set_frame_idx = max(0, min_angle_idx - 20)
    return set_frame_idx

In [ ]:
def normalize_pose(pose_frame):
    normalized = pose_frame.copy()
    valid_x = pose_frame[:, 0][pose_frame[:, 0] > 0]
    valid_y = pose_frame[:, 1][pose_frame[:, 1] > 0]
    
    if len(valid_x) > 0:
        x_min, x_max = np.min(valid_x), np.max(valid_x)
        if x_max > x_min:
            mask_x = pose_frame[:, 0] > 0
            normalized[mask_x, 0] = (pose_frame[mask_x, 0] - x_min) / (x_max - x_min)
    
    if len(valid_y) > 0:
        y_min, y_max = np.min(valid_y), np.max(valid_y)
        if y_max > y_min:
            mask_y = pose_frame[:, 1] > 0
            normalized[mask_y, 1] = (pose_frame[mask_y, 1] - y_min) / (y_max - y_min)
    return normalized

In [ ]:
def extract_features(pose_sequence):
    set_frame_idx = find_set_position_frame(pose_sequence)
    set_frame = pose_sequence[set_frame_idx].copy()
    set_frame_norm = normalize_pose(set_frame)
    pose_features = set_frame_norm.flatten()
    mask = np.ones(len(pose_features), dtype=bool)
    pose_features_clean = pose_features[mask]
    
    arm_features = []
    for idx in [2, 3, 4]:
        if set_frame_norm[idx, 0] > 0 and set_frame_norm[idx, 1] > 0:
            arm_features.extend(set_frame_norm[idx])
        else:
            arm_features.extend([0, 0])
    arm_features = np.array(arm_features)
    
    features = {
        "pose_only": pose_features_clean,
        "arms_only": arm_features
    }
    return features, set_frame_idx

In [ ]:
def prepare_dataset(poses, labels):
    X_dict = {"pose_only": [], "arms_only": []}
    set_indices = []
    
    for pose_seq in poses:
        features, set_idx = extract_features(pose_seq)
        set_indices.append(set_idx)
        for key in X_dict.keys():
            X_dict[key].append(features[key])
    
    for key in X_dict.keys():
        X_dict[key] = np.array(X_dict[key])
    return X_dict["pose_only"], set_indices

In [ ]:
poses, labels = loaddir("pitcher_clips/Tyler_Glasnow/keypoints/")
X, set_indices = prepare_dataset(poses, labels)
res = train(X, np.array(labels))